# Bi-Encoder and Cross-Encoder

This notebook uses:

- A **bi-encoder** to retrieve similar documents
- A **cross-encoder** to rerank those documents

No API key is required.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import pandas as pd

# Change only this query and document list to try your own examples.
query = "Which medicine does not cause drowsiness?"

documents = [
    "This medicine commonly causes drowsiness and should be taken at night.",
    "This is a non-drowsy medicine that can be taken safely during the daytime.",
    "The medicine is available as tablets and syrup for adults and children.",
    "Drowsiness is a common side effect of several allergy medicines.",
    "This medicine begins working within thirty minutes."
]

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 1. Bi-encoder retrieval

The query and documents are encoded separately, then compared using cosine similarity.

In [ ]:
# FIRST MAIN CALL: bi-encoder ranking using cosine similarity
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)
document_embeddings = bi_encoder.encode(documents, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, document_embeddings)[0]
top_results = scores.topk(k=len(documents))

candidates = [
    {
        "document": documents[int(index)],
        "bi_encoder_score": float(score)
    }
    for score, index in zip(top_results.values, top_results.indices)
]

bi_ranking = pd.DataFrame(candidates)
bi_ranking.index = range(1, len(bi_ranking) + 1)
bi_ranking.index.name = "bi_encoder_rank"
bi_ranking


,document,bi_encoder_score
bi_encoder_rank,,
1,Drowsiness is a common side effect of several ...,0.729340
2,This medicine commonly causes drowsiness and s...,0.709268
3,This is a non-drowsy medicine that can be take...,0.628850
4,The medicine is available as tablets and syrup...,0.395035
5,This medicine begins working within thirty min...,0.339929


## 2. Cross-encoder reranking

The cross-encoder reads the query and each candidate together and produces a relevance score.

In [ ]:
# SECOND MAIN CALL: cross-encoder reranking
pairs = [[query, item["document"]] for item in candidates]
cross_scores = cross_encoder.predict(pairs)

for item, score in zip(candidates, cross_scores):
    item["cross_encoder_score"] = float(score)

reranked = sorted(
    candidates,
    key=lambda item: item["cross_encoder_score"],
    reverse=True
)

comparison = pd.DataFrame(reranked)
comparison["cross_encoder_rank"] = range(1, len(comparison) + 1)

# Add the original bi-encoder rank for comparison.
bi_rank_lookup = {
    item["document"]: rank
    for rank, item in enumerate(candidates, start=1)
}
comparison["bi_encoder_rank"] = comparison["document"].map(bi_rank_lookup)

comparison = comparison[
    [
        "document",
        "bi_encoder_rank",
        "bi_encoder_score",
        "cross_encoder_rank",
        "cross_encoder_score"
    ]
]

comparison


,document,bi_encoder_rank,bi_encoder_score,cross_encoder_rank,cross_encoder_score
0,This medicine commonly causes drowsiness and s...,2,0.709268,1,5.028437
1,Drowsiness is a common side effect of several ...,1,0.729340,2,3.854576
2,This is a non-drowsy medicine that can be take...,3,0.628850,3,2.872033
3,The medicine is available as tablets and syrup...,4,0.395035,4,-8.921837
4,This medicine begins working within thirty min...,5,0.339929,5,-9.409506


## What to observe

The sentence containing **"causes drowsiness"** may receive a strong bi-encoder score because it shares many words with the query.

The cross-encoder examines the query and document together and is better able to recognize that the query asks for a medicine that **does not** cause drowsiness.

```text
Bi-encoder → retrieves using overall semantic similarity
Cross-encoder → reranks using detailed query-document interaction
```

Bi-encoder scores are cosine similarities. Cross-encoder scores are raw relevance scores called logits.


## Hybrid Search: BM25 + Bi-Encoder using RRF

BM25 uses keyword matching, while the bi-encoder uses semantic similarity.
Reciprocal Rank Fusion (RRF) combines both rankings.


In [ ]:
!pip install -q rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Sparse ranking: exact keyword matching
bm25 = BM25Okapi([doc.lower().split() for doc in documents])
bm25_scores = np.array(bm25.get_scores(query.lower().split()))

In [ ]:
bm25_scores

array([0.20930251, 0.19317475, 0.2009155 , 0.        , 0.25125624])

In [ ]:
list(np.argsort(-bm25_scores))

[np.int64(4), np.int64(0), np.int64(2), np.int64(1), np.int64(3)]

In [ ]:

# Dense ranking: semantic similarity
dense_scores = util.cos_sim(
    query_embedding,
    document_embeddings
)[0].cpu().numpy()

def ranking(scores):
    return list(np.argsort(-scores))

def reciprocal_rank_fusion(rankings, k=60):
    fused_scores = {}

    for result_ranking in rankings:
        for rank, document_index in enumerate(result_ranking):
            fused_scores[document_index] = (
                fused_scores.get(document_index, 0.0) + 1.0 / (k + rank + 1)
            )

    return fused_scores

fused = reciprocal_rank_fusion([
    ranking(bm25_scores),
    ranking(dense_scores)
])

hybrid_results = pd.DataFrame([
    {
        "document": documents[index],
        "bm25_score": round(float(bm25_scores[index]), 4),
        "dense_score": round(float(dense_scores[index]), 4),
        "rrf_score": round(float(score), 5)
    }
    for index, score in sorted(
        fused.items(),
        key=lambda item: item[1],
        reverse=True
    )
])

hybrid_results.index = range(1, len(hybrid_results) + 1)
hybrid_results.index.name = "hybrid_rank"
hybrid_results


,document,bm25_score,dense_score,rrf_score
hybrid_rank,,,,
1,This medicine commonly causes drowsiness and s...,0.2093,0.7093,0.03226
2,This medicine begins working within thirty min...,0.2513,0.3399,0.03178
3,Drowsiness is a common side effect of several ...,0.0000,0.7293,0.03178
4,The medicine is available as tablets and syrup...,0.2009,0.3950,0.03150
5,This is a non-drowsy medicine that can be take...,0.1932,0.6288,0.03150


### Simple explanation

- **BM25** finds documents with matching words.
- **Bi-encoder** finds documents with similar meaning.
- **RRF** combines their rank positions.
- A document ranked highly by either method gets a higher final score.

RRF uses rank positions instead of directly adding BM25 and cosine scores because those scores use different scales.
